In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.api as sm

from optbinning import OptimalBinning
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
FILE_PATH        = r"Dissertation_Data\Processed\master_data_flagged_trainval.parquet"
ID_COL           = "SK_ID_CURR"
TARGET_COL       = "TARGET"
#STRATIFY_COL     = "CODE_GENDER"

MISSING_THR      = 0.20
INF_THR          = 0.20
IV_THR           = 0.01
VIF_THR          = 10          # VIF threshold (replaces GVIF_THR)
CORR_THR         = 0.80
GINI_IMPROVE_THR = 0.001
TEST_SIZE        = 0.30
RANDOM_STATE     = 42
IV_BINS          = 10
EPS              = 1e-6

OUTPUT_CSV          = "master_data_train_test_split.csv"
FEATURE_SHORTLIST   = "feature_shortlist.csv"

## 1. Load data — Train/Test split & save CSV with flag_train_test

In [ ]:
df = pd.read_parquet(FILE_PATH)
df = df.drop(columns=[c for c in ["flag_train_val"] if c in df.columns])
print("Shape after load:", df.shape)

In [ ]:
# separate features, target
list_features = [col for col in df.columns if col not in ['TARGET','SK_ID_CURR']]
# features that starts with EXT_
list_ext_features = [col for col in df.columns if col.startswith('EXT_')]
# list ethical features
list_ethical_features = [col for col in df.columns if 'GENDER' in col]

In [ ]:
cols_to_drop = list(set(list_ext_features + list_ethical_features))
df = df.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
print("Shape after dropping EXT_ and GENDER columns:", df.shape)

In [ ]:
# ── Train / Test split right at Step 1 ────────────────────────────────────────
train_idx, test_idx = train_test_split(
    df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[TARGET_COL],
)

df["flag_train_test"] = "test"
df.loc[train_idx, "flag_train_test"] = "train"

# Save full data with flag to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")
print(f"Train: {(df['flag_train_test']=='train').sum():,}  |  Test: {(df['flag_train_test']=='test').sum():,}")
print("Shape:", df.shape)

## 2. Drop high-missing & high-infinity features

In [ ]:
# Work on df without the flag column for feature selection
flag_col = df["flag_train_test"].copy()
df = df.drop(columns=["flag_train_test"])

# Missing
miss_rate    = df.isna().mean()
drop_missing = miss_rate[miss_rate > MISSING_THR].index.tolist()
df           = df.drop(columns=drop_missing)

# Infinity (numeric only)
num_cols  = df.select_dtypes(include=np.number).columns
inf_rate  = np.isinf(df[num_cols]).mean()
drop_inf  = inf_rate[inf_rate > INF_THR].index.tolist()
df        = df.drop(columns=drop_inf)

# Re-attach flag
df["flag_train_test"] = flag_col

print(f"Dropped missing : {len(drop_missing)}")
print(f"Dropped infinity: {len(drop_inf)}")
print("Shape:", df.shape)

## 3. Information Value (IV) via AutoBinning on full data — then compute df_woe

In [ ]:
features = [c for c in df.columns if c not in [ID_COL, TARGET_COL, "flag_train_test"]]
y_all    = df[TARGET_COL].values

# ── Fit OptimalBinning on full data, extract IV and WoE in one pass ────────────
binners   = {}   # fitted OptimalBinning objects
woe_cols  = {}   # WoE-transformed arrays
iv_rows   = []   # IV per feature

for col in features:
    dtype = "numerical" if pd.api.types.is_numeric_dtype(df[col]) else "categorical"
    ob    = OptimalBinning(name=col, dtype=dtype, solver="cp", max_n_bins=IV_BINS)
    ob.fit(df[col].values, y_all)
    binners[col]  = ob
    woe_cols[col] = ob.transform(df[col].values, metric="woe")

    # Extract IV from binning table
    try:
        bt  = ob.binning_table.build()
        iv  = bt.loc[bt.index != "Totals", "IV"].sum()
    except Exception:
        iv = np.nan
    iv_rows.append({"feature": col, "iv": iv})

# ── IV table ───────────────────────────────────────────────────────────────────
iv_table = (
    pd.DataFrame(iv_rows)
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)

bins_iv = [-np.inf, 0.01, 0.02, 0.10, 0.30, 0.50, np.inf]
labels  = ["<0.01", "0.01-0.02", "0.02-0.10", "0.10-0.30", "0.30-0.50", ">0.50"]
iv_table["group"] = pd.cut(iv_table["iv"], bins=bins_iv, labels=labels)
print(iv_table["group"].value_counts().reindex(labels).fillna(0).astype(int))

# ── Filter by IV ───────────────────────────────────────────────────────────────
keep_iv   = iv_table.loc[iv_table.iv >= IV_THR, "feature"].tolist()
print(f"\nFeatures after IV filter: {len(keep_iv)}")

# ── Build df_woe on full data (IV-passing features only) ──────────────────────
df_woe = pd.DataFrame({col: woe_cols[col] for col in keep_iv}, index=df.index)
df_woe.insert(0, ID_COL,          df[ID_COL].values)
df_woe.insert(1, TARGET_COL,      df[TARGET_COL].values)
df_woe.insert(2, "flag_train_test", df["flag_train_test"].values)

print("df_woe shape:", df_woe.shape)
df_woe.head(3)

## 4. Correlation on df_woe — keep higher-IV feature in each correlated pair

In [ ]:
woe_features = [c for c in df_woe.columns if c not in [ID_COL, TARGET_COL, "flag_train_test"]]

corr = df_woe[woe_features].corr().abs()

# Collect highly-correlated pairs
pairs = [
    {"feature_1": corr.columns[i], "feature_2": corr.columns[j], "corr": corr.iloc[i, j]}
    for i in range(len(corr.columns))
    for j in range(i + 1, len(corr.columns))
    if corr.iloc[i, j] > CORR_THR
]
pairs = pd.DataFrame(pairs).sort_values("corr", ascending=False).reset_index(drop=True)
print(f"High-corr pairs (>{CORR_THR}): {len(pairs)}")

# Drop lower-IV feature in each pair
iv_dict  = iv_table.set_index("feature")["iv"].to_dict()
drop_set = set()
for _, row in pairs.iterrows():
    f1, f2 = row.feature_1, row.feature_2
    if f1 in drop_set or f2 in drop_set:
        continue
    drop_set.add(f2 if iv_dict.get(f1, 0) >= iv_dict.get(f2, 0) else f1)

df_woe = df_woe.drop(columns=[c for c in drop_set if c in df_woe.columns])
print(f"Dropped by correlation: {len(drop_set)}")
print("df_woe shape after corr filter:", df_woe.shape)

# Refresh IV table to surviving features
surviving = [c for c in df_woe.columns if c not in [ID_COL, TARGET_COL, "flag_train_test"]]
iv_table  = (
    iv_table[iv_table.feature.isin(surviving)]
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)

## 5. VIF on df_woe — drop multicollinear features

In [ ]:
import time

woe_features = [c for c in df_woe.columns if c not in [ID_COL, TARGET_COL, "flag_train_test"]]

# df_woe columns are already numeric WoE values — impute any residual NaN
X_vif = df_woe[woe_features].copy()
X_vif = X_vif.replace([np.inf, -np.inf], np.nan).fillna(X_vif.median())

print(f"Calculating VIF for {X_vif.shape[1]} WoE features...")
t0 = time.time()

# Iterative VIF elimination
remaining = woe_features.copy()
drop_vif  = []

while True:
    X_tmp = sm.add_constant(X_vif[remaining], has_constant="add")
    vif_vals = [
        variance_inflation_factor(X_tmp.values, i + 1)  # +1 to skip constant
        for i in range(len(remaining))
    ]
    vif_series = pd.Series(vif_vals, index=remaining)
    max_vif    = vif_series.max()

    if max_vif <= VIF_THR:
        break

    worst = vif_series.idxmax()
    print(f"  Drop '{worst}'  VIF={max_vif:.2f}")
    drop_vif.append(worst)
    remaining.remove(worst)

print(f"Finished VIF in {time.time() - t0:.1f}s")

vif_table = pd.DataFrame(
    {"feature": remaining,
     "vif": [variance_inflation_factor(
                 sm.add_constant(X_vif[remaining], has_constant="add").values, i + 1
             ) for i in range(len(remaining))]}
).sort_values("vif", ascending=False).reset_index(drop=True)

print(vif_table.to_string(index=False))
print(f"\nDropped by VIF: {len(drop_vif)}  →  {drop_vif}")

df_woe = df_woe.drop(columns=[c for c in drop_vif if c in df_woe.columns])
print("df_woe shape after VIF filter:", df_woe.shape)

# Refresh IV table
surviving = [c for c in df_woe.columns if c not in [ID_COL, TARGET_COL, "flag_train_test"]]
iv_table  = (
    iv_table[iv_table.feature.isin(surviving)]
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)

## 6. Save feature shortlist before stepwise

In [ ]:
shortlist_features = [c for c in df_woe.columns if c not in [ID_COL, TARGET_COL, "flag_train_test"]]

shortlist_df = (
    iv_table[iv_table.feature.isin(shortlist_features)]
    [["feature", "iv", "group"]]
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)
shortlist_df.to_csv(FEATURE_SHORTLIST, index=False)

print(f"Feature shortlist saved to: {FEATURE_SHORTLIST}")
print(f"Total features in shortlist: {len(shortlist_features)}")
print(shortlist_df.to_string(index=False))

## 7. Train / Test split from df_woe

In [ ]:
train_woe = df_woe[df_woe["flag_train_test"] == "train"].reset_index(drop=True)
test_woe  = df_woe[df_woe["flag_train_test"] == "test"].reset_index(drop=True)

model_features = [c for c in df_woe.columns if c not in [ID_COL, TARGET_COL, "flag_train_test"]]

X_train = train_woe[model_features]
y_train = train_woe[TARGET_COL]
X_test  = test_woe[model_features]
y_test  = test_woe[TARGET_COL]

print(f"Train: {train_woe.shape}  |  Test: {test_woe.shape}")

## 8. Stepwise Logistic Regression (forward, Gini-based)

In [ ]:
from numpy.linalg import matrix_rank

def fit_logit(X, y):
    X = X.replace([np.inf, -np.inf], np.nan)
    if X.isna().any().any():
        raise ValueError("X contains NaN.")
    X_c = sm.add_constant(X, has_constant="add")
    if matrix_rank(X_c) < X_c.shape[1]:
        return None
    return sm.Logit(y, X_c).fit(disp=False, method="newton")


def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1


# Use shortlist order (sorted by IV) as candidates
candidates   = [f for f in shortlist_df["feature"].tolist() if f in X_train.columns]
selected     = []
current_gini = 0.0
log_rows     = []

for i, feat in enumerate(candidates, 1):
    print(f"[{i}/{len(candidates)}] Testing: {feat}", flush=True)

    trial_feats = selected + [feat]
    model_try   = fit_logit(X_train[trial_feats], y_train)

    if model_try is None:
        print("    -> skipped (singular matrix)", flush=True)
        log_rows.append({"feature": feat, "gini_train": None,
                         "improvement": None, "accepted": False, "note": "singular"})
        continue

    prob_train  = model_try.predict(sm.add_constant(X_train[trial_feats], has_constant="add"))
    new_gini    = gini(y_train, prob_train)
    improvement = new_gini - current_gini
    accept      = improvement >= GINI_IMPROVE_THR

    print(f"    Gini={new_gini:.4f}  Improve={improvement:.4f}  {'ACCEPT' if accept else 'REJECT'}",
          flush=True)

    log_rows.append({"feature": feat, "gini_train": round(new_gini, 4),
                     "improvement": round(improvement, 4), "accepted": accept, "note": ""})

    if accept:
        selected     = trial_feats
        current_gini = new_gini

step_log = pd.DataFrame(log_rows)
print(step_log.to_string(index=False))
print(f"\nFinal model features ({len(selected)}): {selected}")

## 9. Final model & Test Gini

In [ ]:
final_model = fit_logit(X_train[selected], y_train)
print(final_model.summary())

prob_train_final = final_model.predict(sm.add_constant(X_train[selected]))
prob_test_final  = final_model.predict(sm.add_constant(X_test[selected]))

gini_train = gini(y_train, prob_train_final)
gini_test  = gini(y_test,  prob_test_final)

print(f"\nGini Train : {gini_train:.4f}")
print(f"Gini Test  : {gini_test:.4f}")